# Tahap 2: Case Representation
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Domain:** Perlindungan Anak  
**Tujuan:** Ekstraksi metadata & fitur teks dari corpus putusan, simpan ke `cases.csv` dan `cases.json`

### Alur Notebook:
1. Install & Import
2. Konfigurasi & Load Data dari Tahap 1
3. Koneksi Groq API
4. Ekstraksi Metadata per Dokumen (via Groq + fallback regex)
5. Feature Engineering
6. Simpan ke CSV & JSON
7. Validasi & Preview Hasil

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q groq pandas tqdm

## Cell 2 — Import Library

In [25]:
import os
import re
import json
import time
import logging
from pathlib import Path
from getpass import getpass

import pandas as pd
from tqdm import tqdm
from groq import Groq

print("✅ Library berhasil diimport")

✅ Library berhasil diimport


## Cell 3 — Konfigurasi

In [26]:
# ============================================================
# KONFIGURASI — sesuaikan dengan hasil Tahap 1
# ============================================================
CONFIG = {
    "DOMAIN"          : "Perlindungan Anak",
    "GROQ_MODEL"      : "llama-3.3-70b-versatile",  # model gratis Groq
    "GROQ_DELAY"      : 1.5,   # detik antar request (hindari rate limit)
    "MAX_CHARS_INPUT" : 6000,  # batas karakter teks yang dikirim ke Groq
    "TOP_K_PREVIEW"   : 5,
    "MIN_DOKUMEN"     : 5,     # target minimum dokumen (5 untuk testing, 30 untuk final)     # jumlah preview di akhir
}

# Direktori (harus sama dengan Tahap 1)
BASE_DIR   = Path("../STEP 1").resolve()
DATA_RAW   = BASE_DIR / "data" / "raw"
DATA_PROC  = BASE_DIR / "data" / "processed"
LOGS_DIR   = BASE_DIR / "logs"
META_FILE  = BASE_DIR / "data" / "metadata_with_pdf.json"

DATA_PROC.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# Setup logging
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "representation.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print(f"📁 Raw text : {DATA_RAW}")
print(f"📁 Processed: {DATA_PROC}")

📁 Raw text : D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\raw
📁 Processed: D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\processed


## Cell 4 — Load Data dari Tahap 1

In [27]:
# Load metadata dari Tahap 1
assert META_FILE.exists(), f"❌ {META_FILE} tidak ditemukan. Jalankan Tahap 1 dulu."

with open(META_FILE, encoding="utf-8") as f:
    metadata_raw = json.load(f)

# Filter hanya yang berhasil dan punya file teks
dokumen_valid = []
for item in metadata_raw:
    case_id  = item.get("case_id", "")
    path_txt = DATA_RAW / f"{case_id}.txt"
    if item.get("status_pdf") in ("berhasil", "sudah_ada") and path_txt.exists():
        item["path_txt"] = str(path_txt)
        dokumen_valid.append(item)

# Ambil minimum dari CONFIG (default 30 jika tidak ada)
MIN_DOK = CONFIG.get("MIN_DOKUMEN", 30)

print(f"✅ Total dokumen valid dari Tahap 1 : {len(dokumen_valid)}")
print(f"⚠️  Dokumen tanpa teks (skip)        : {len(metadata_raw) - len(dokumen_valid)}")

if len(dokumen_valid) < MIN_DOK:
    print(f"\n⚠️  Perhatian: dokumen valid ({len(dokumen_valid)}) di bawah target ({MIN_DOK}).")
    print(f"   Untuk testing dengan sedikit dokumen, ini normal.")
else:
    print(f"\n✅ Siap diproses!")

✅ Total dokumen valid dari Tahap 1 : 43
⚠️  Dokumen tanpa teks (skip)        : 57

✅ Siap diproses!


## Cell 5 — Koneksi Groq API

> Daftar gratis di https://console.groq.com → pilih **API Keys** → **Create API Key**  
> Paste key-nya di bawah saat diminta. Key tidak akan ditampilkan di layar.

In [28]:
# ============================================================
# MODE TANPA GROQ — ekstraksi metadata pakai regex saja
# ============================================================
client     = None
GROQ_AKTIF = False
print("⏭️  Groq dilewati — ekstraksi metadata 100% pakai regex (offline)")

⏭️  Groq dilewati — ekstraksi metadata 100% pakai regex (offline)


## Cell 6 — Prompt & Fungsi Ekstraksi Metadata

In [29]:
def ekstrak_via_groq(teks: str, case_id: str) -> dict:
    """
    Versi non-aktif (mode tanpa Groq).
    Selalu return dict kosong agar pipeline otomatis pakai regex.
    """
    return {}

print("✅ Groq dinonaktifkan — fungsi ekstraksi akan pakai regex")

✅ Groq dinonaktifkan — fungsi ekstraksi akan pakai regex


## Cell 7 — Fallback Regex

> Dipakai bila Groq gagal (rate limit, koneksi, dll)  
> Pola regex disesuaikan dengan struktur putusan MA RI domain Perlindungan Anak

In [30]:
BULAN_ID = (
    "Januari|Februari|Maret|April|Mei|Juni|"
    "Juli|Agustus|September|Oktober|November|Desember"
)

def first_match(patterns: list, text: str, group: int = 1) -> str:
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE | re.DOTALL)
        if m:
            try:
                val = m.group(group).strip()
            except IndexError:
                val = m.group(0).strip()
            return re.sub(r"\s+", " ", val)
    return ""


def bersihkan_teks_disclaimer(teks: str) -> str:
    """
    Hapus semua baris disclaimer MA RI yang berulang,
    sisakan hanya konten aktual putusan.
    """
    pola_buang = [
        r"disclaimer",
        r"pelaksanaan fungsi peradilan",
        r"kepaniteraan@mahkamahagung",
        r"email\s*:",
        r"telp\s*:",
        r"^halaman\s+\d+$",
        r"hal mana akan terus kami perbaiki",
        r"akurasi dan keterkinian",
    ]
    baris_bersih = []
    for baris in teks.split("\n"):
        b = baris.strip()
        if not b:
            continue
        buang = any(re.search(p, b, re.IGNORECASE) for p in pola_buang)
        if not buang:
            baris_bersih.append(b)
    return " ".join(baris_bersih)


def ekstrak_no_perkara_dari_judul(judul: str) -> str:
    """Ekstrak nomor perkara dari field judul metadata."""
    m = re.search(
        r"(?:Nomor|No\.?)\s+([\d\s/.\-]+(?:Pid|Pdt)[.\w\s\-/]+\d{4}[^\s,]*)",
        judul, re.IGNORECASE
    )
    if m:
        return re.sub(r"\s+", " ", m.group(1).strip())
    # Fallback: ambil pola nomor langsung
    m2 = re.search(r"(\d+\s*/\s*(?:Pid|Pdt)[.\w\s\-/]+/\s*(?:PN|PA)\s*\w+)", judul, re.IGNORECASE)
    if m2:
        return re.sub(r"\s+", " ", m2.group(1).strip())
    return ""


def ekstrak_via_regex(teks: str, judul: str = "", url: str = "") -> dict:
    """
    Ekstrak metadata dari teks putusan MA RI.
    Karena PDF terproteksi dan hanya halaman terakhir yang terbaca,
    sebagian metadata diambil dari judul dan URL.
    """
    # Bersihkan disclaimer dulu
    teks_bersih = bersihkan_teks_disclaimer(teks)

    # Nomor perkara — utamakan dari judul karena lebih lengkap
    no_perkara = ekstrak_no_perkara_dari_judul(judul)
    if not no_perkara:
        no_perkara = first_match([
            r"(?:Nomor|No\.?)\s*[:\.\-]?\s*(\d+[/\-][A-Za-z.]+[/\-]\d{4}[/\-][A-Z.\s]+)",
            r"Putusan\s+(?:Pidana|Perdata)?\s*(?:Nomor|No\.?)?\s*([\d\s/.\-]+(?:Pid|Pdt)[.\w\s\-/]+\d{4}[^\s,]*)",
            r"Penetapan\s+Nomor\s+([\d\s/.\-]+(?:Pid|Pdt)[.\w\s\-/]+\d{4}[^\s,]*)",
        ], teks_bersih)

    # Tanggal — cari di teks bersih
    tanggal = first_match([
        r"(?:tanggal|pada|ditetapkan\s+di[^,]*,\s*(?:pada\s+)?tanggal)\s*[:\-]?\s*(\d{1,2}\s+(?:" + BULAN_ID + r")\s+\d{4})",
        r"(\d{1,2}\s+(?:" + BULAN_ID + r")\s+\d{4})",
        r"(\d{4}-\d{2}-\d{2})",
    ], teks_bersih)

    # Pengadilan — selalu PN Denpasar untuk dataset ini
    pengadilan = first_match([
        r"(Pengadilan\s+Negeri\s+\w+)",
        r"PN\s+(\w+)",
    ], teks_bersih)
    if not pengadilan and ("PN Dps" in judul or "PN.DPS" in judul.upper()):
        pengadilan = "Pengadilan Negeri Denpasar"

    # Terdakwa
    terdakwa = first_match([
        r"(?:Terdakwa|TERDAKWA|Anak)\s*[:\-]?\s*([A-Z][A-Za-z\s,./]+?)(?=\s*(?:bin|binti|alias|,|\n|;))",
        r"dihadapan\s+(?:Terdakwa\s+)?([A-Z][A-Za-z\s]+?)(?=\s*(?:dan|dengan|,|\n))",
    ], teks_bersih)

    # Jaksa
    jaksa = first_match([
        r"(?:Jaksa\s+Penuntut\s+Umum|JPU)\s+(?:pada\s+\w+\s+Negeri\s+\w+\s+)?([A-Z][A-Za-z\s,.]+?)(?=\s*(?:dan|serta|,|\n|;))",
        r"dihadiri\s+oleh\s*[:\-]?\s*([A-Z][A-Za-z\s,.]+?)\s*(?:SH|,\s*Jaksa)",
    ], teks_bersih)

    # Amar putusan — cari kata kunci keputusan
    amar = first_match([
        r"((?:menyatakan\s+menerima|pikir-pikir|terbukti|tidak\s+terbukti|menghentikan|dibebaskan)[^.;]{0,200})",
        r"((?:Menghukum|Menjatuhkan|Memutuskan|MENGADILI)[^.]{0,200})",
        r"menyatakan\s+(menerima\s+(?:baik\s+)?[Pp]utusan[^.]{0,150})",
    ], teks_bersih)

    # Vonis penjara
    vonis = first_match([
        r"(\d+\s*(?:tahun|bulan)\s*(?:dan\s*\d+\s*(?:tahun|bulan))?)\s*(?:penjara|pidana\s+penjara)",
        r"pidana\s+(?:penjara\s+)?selama\s+(\d+[^;.]{0,30})",
    ], teks_bersih)

    # Denda
    denda = first_match([
        r"denda\s+(?:sebesar\s+)?(?:Rp\.?\s*)?([\d.,]+[^;.\n]{0,30})",
        r"membayar\s+biaya\s+perkara\s+(?:sebesar\s+)?(?:Rp\.?\s*)?([\d.,]+[^\n]{0,20})",
    ], teks_bersih)

    # Jenis dokumen (putusan / penetapan / kesepakatan diversi)
    jenis_dok = "putusan"
    if "penetapan" in teks_bersih.lower()[:100] or "penetapan" in judul.lower():
        jenis_dok = "penetapan"
    elif "diversi" in teks_bersih.lower() or "diversi" in judul.lower():
        jenis_dok = "kesepakatan_diversi"

    return {
        "no_perkara"     : no_perkara,
        "tanggal_putusan": tanggal,
        "pengadilan"     : pengadilan,
        "terdakwa"       : terdakwa,
        "jaksa"          : jaksa,
        "pasal_dakwaan"  : "",   # tidak tersedia dari halaman terakhir
        "pasal_terbukti" : "",
        "amar_putusan"   : amar,
        "vonis_penjara"  : vonis,
        "vonis_denda"    : denda,
        "jenis_dokumen"  : jenis_dok,
        "barang_bukti"   : "",
        "ringkasan_fakta": teks_bersih[:300] if teks_bersih else "",
    }

print("✅ Fungsi fallback regex (versi PDF terproteksi) siap")

✅ Fungsi fallback regex (versi PDF terproteksi) siap


## Cell 8 — Feature Engineering

In [31]:
def hitung_fitur(teks: str) -> dict:
    """
    Hitung fitur statistik sederhana dari teks putusan.
    Dipakai sebagai tambahan representasi di samping metadata.
    """
    kata       = teks.split()
    kalimat    = re.split(r"[.!?]+", teks)
    paragraf   = [p for p in teks.split("\n\n") if p.strip()]

    # Deteksi apakah ada kata kunci perlindungan anak
    kata_kunci_pa = [
        "anak", "korban", "kekerasan", "seksual", "eksploitasi",
        "trafficking", "perdagangan", "perlindungan", "minor",
        "di bawah umur", "pencabulan", "pemerkosaan", "penganiayaan"
    ]
    teks_lower = teks.lower()
    kata_kunci_ditemukan = [k for k in kata_kunci_pa if k in teks_lower]

    return {
        "jumlah_kata"      : len(kata),
        "jumlah_kalimat"   : len([k for k in kalimat if k.strip()]),
        "jumlah_paragraf"  : len(paragraf),
        "avg_kata_kalimat" : round(len(kata) / max(len(kalimat), 1), 1),
        "kata_kunci_pa"    : ", ".join(kata_kunci_ditemukan),
        "n_kata_kunci_pa"  : len(kata_kunci_ditemukan),
    }

print("✅ Fungsi feature engineering siap")

✅ Fungsi feature engineering siap


## Cell 9 — Pipeline Utama: Ekstraksi + Feature Engineering

In [32]:
def pipeline_tahap2(dokumen_list: list) -> list:
    """
    Pipeline lengkap Tahap 2:
    Teks bersih → Groq/regex → metadata + fitur → list of dict

    Setiap record berisi:
    - identitas (case_id, no_perkara, tanggal, dst)
    - metadata hukum (pasal, terdakwa, amar, vonis, dst)
    - fitur teks (jumlah kata, kata kunci, dst)
    - teks_full (teks lengkap, untuk embedding di Tahap 3)
    """
    records  = []
    n_groq   = 0
    n_regex  = 0
    n_gagal  = 0

    print(f"🔄 Memproses {len(dokumen_list)} dokumen...\n")

    for item in tqdm(dokumen_list, desc="Ekstraksi"):
        case_id  = item["case_id"]
        path_txt = Path(item["path_txt"])

        # Baca teks
        try:
            teks = path_txt.read_text(encoding="utf-8", errors="ignore").strip()
        except Exception as e:
            logger.error(f"{case_id}: gagal baca teks — {e}")
            n_gagal += 1
            continue

        if not teks or len(teks.split()) < 50:
            logger.warning(f"{case_id}: teks terlalu pendek, skip")
            n_gagal += 1
            continue

        # ── Ekstraksi metadata ──────────────────────────────────────
        metadata = ekstrak_via_groq(teks, case_id)

        if not metadata or not any(metadata.values()):
            # Fallback ke regex
            logger.warning(f"{case_id}: Groq gagal/kosong → pakai regex")
            metadata = ekstrak_via_regex(teks)
            n_regex += 1
        else:
            n_groq += 1

        # Delay hanya jika pakai Groq (mode regex tidak perlu delay)
        if GROQ_AKTIF:
            time.sleep(CONFIG["GROQ_DELAY"])

        # ── Feature engineering ─────────────────────────────────────
        fitur = hitung_fitur(teks)

        # ── Susun record ────────────────────────────────────────────
        record = {
            # Identitas
            "case_id"          : case_id,
            "url_detail"       : item.get("url_detail", ""),
            "path_txt"         : str(path_txt),

            # Metadata dari Groq/regex
            "no_perkara"       : metadata.get("no_perkara", ""),
            "tanggal_putusan"  : metadata.get("tanggal_putusan", ""),
            "pengadilan"       : metadata.get("pengadilan", ""),
            "terdakwa"         : metadata.get("terdakwa", ""),
            "jaksa"            : metadata.get("jaksa", ""),
            "pasal_dakwaan"    : metadata.get("pasal_dakwaan", ""),
            "pasal_terbukti"   : metadata.get("pasal_terbukti", ""),
            "amar_putusan"     : metadata.get("amar_putusan", ""),
            "vonis_penjara"    : metadata.get("vonis_penjara", ""),
            "vonis_denda"      : metadata.get("vonis_denda", ""),
            "barang_bukti"     : metadata.get("barang_bukti", ""),
            "ringkasan_fakta"  : metadata.get("ringkasan_fakta", ""),

            # Fitur teks
            "jumlah_kata"      : fitur["jumlah_kata"],
            "jumlah_kalimat"   : fitur["jumlah_kalimat"],
            "jumlah_paragraf"  : fitur["jumlah_paragraf"],
            "avg_kata_kalimat" : fitur["avg_kata_kalimat"],
            "kata_kunci_pa"    : fitur["kata_kunci_pa"],
            "n_kata_kunci_pa"  : fitur["n_kata_kunci_pa"],

            # Sumber ekstraksi
            "sumber_ekstraksi" : "regex",

            # Teks lengkap (dibutuhkan Tahap 3 untuk embedding)
            "teks_full"        : teks,
        }
        records.append(record)

    print(f"\n{'='*50}")
    print(f"  REKAP PIPELINE TAHAP 2")
    print(f"{'='*50}")
    print(f"  ✅ Total record    : {len(records)}")
    print(f"  🤖 Via Groq        : {n_groq}")
    print(f"  🔤 Via regex       : {n_regex}")
    print(f"  ❌ Gagal           : {n_gagal}")
    print(f"{'='*50}")

    return records

print("✅ Fungsi pipeline_tahap2 siap")

✅ Fungsi pipeline_tahap2 siap


In [33]:
# Jalankan pipeline
records = pipeline_tahap2(dokumen_valid)

🔄 Memproses 43 dokumen...



Ekstraksi:   0%|          | 0/43 [00:00<?, ?it/s]2026-06-13 22:24:09,239 | WARNING | case_047: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,252 | WARNING | case_048: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,291 | WARNING | case_049: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,320 | WARNING | case_050: Groq gagal/kosong → pakai regex
Ekstraksi:   9%|▉         | 4/43 [00:00<00:01, 35.42it/s]2026-06-13 22:24:09,352 | WARNING | case_051: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,368 | WARNING | case_052: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,387 | WARNING | case_053: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,404 | WARNING | case_054: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,423 | WARNING | case_055: Groq gagal/kosong → pakai regex
Ekstraksi:  21%|██        | 9/43 [00:00<00:00, 43.35it/s]2026-06-13 22:24:09,453 | WARNING | case_057: Groq gagal/kosong → pakai regex
2026-06-13 22:24:09,464 | WARNING | case_058: Groq gagal/


  REKAP PIPELINE TAHAP 2
  ✅ Total record    : 43
  🤖 Via Groq        : 0
  🔤 Via regex       : 43
  ❌ Gagal           : 0


## Cell 10 — Simpan ke CSV & JSON

In [ ]:
assert records, "❌ Tidak ada record — jalankan Cell 9 dulu"

# ── Simpan CSV ──────────────────────────────────────────────
# CSV tanpa kolom teks_full (terlalu besar, ada di JSON)
df = pd.DataFrame(records)
kolom_csv = [c for c in df.columns if c != "teks_full"]
df_csv    = df[kolom_csv]

csv_path = DATA_PROC / "cases.csv"
df_csv.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ CSV disimpan: {csv_path}  ({len(df_csv)} baris, {len(kolom_csv)} kolom)")

# ── Simpan JSON (lengkap dengan teks_full) ──────────────────
json_path = DATA_PROC / "cases.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"✅ JSON disimpan: {json_path}  ({len(records)} record)")

# Ukuran file
print(f"\n  Ukuran CSV  : {csv_path.stat().st_size / 1024:.1f} KB")
print(f"  Ukuran JSON : {json_path.stat().st_size / 1024:.1f} KB")

✅ CSV disimpan: D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\processed\cases.csv  (43 baris, 22 kolom)
✅ JSON disimpan: D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\processed\cases.json  (43 record)

  Ukuran CSV  : 40.5 KB
  Ukuran JSON : 2456.2 KB


## Cell 11 — Validasi Kelengkapan Metadata

In [35]:
df = pd.read_csv(DATA_PROC / "cases.csv")

print(f"{'='*55}")
print(f"  LAPORAN VALIDASI — TAHAP 2")
print(f"{'='*55}")
print(f"  Total kasus   : {len(df)}")
print(f"{'='*55}")

# Cek kelengkapan per kolom metadata penting
kolom_penting = [
    "no_perkara", "tanggal_putusan", "pengadilan",
    "terdakwa", "pasal_dakwaan", "amar_putusan",
    "vonis_penjara", "ringkasan_fakta"
]

print(f"\n  Kelengkapan metadata:")
for k in kolom_penting:
    if k in df.columns:
        terisi = df[k].astype(str).str.strip().ne("").sum()
        pct    = terisi / len(df) * 100
        bar    = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"    {k:<20} [{bar}] {pct:5.1f}%  ({terisi}/{len(df)})")

print(f"\n  Sumber ekstraksi:")
if "sumber_ekstraksi" in df.columns:
    print(df["sumber_ekstraksi"].value_counts().to_string())

print(f"\n  Statistik jumlah kata:")
print(f"    Min  : {df['jumlah_kata'].min():,}")
print(f"    Max  : {df['jumlah_kata'].max():,}")
print(f"    Rata : {df['jumlah_kata'].mean():.0f}")
print(f"{'='*55}")

  LAPORAN VALIDASI — TAHAP 2
  Total kasus   : 43

  Kelengkapan metadata:
    no_perkara           [██████████] 100.0%  (43/43)
    tanggal_putusan      [██████████] 100.0%  (43/43)
    pengadilan           [██████████] 100.0%  (43/43)
    terdakwa             [██████████] 100.0%  (43/43)
    pasal_dakwaan        [██████████] 100.0%  (43/43)
    amar_putusan         [██████████] 100.0%  (43/43)
    vonis_penjara        [██████████] 100.0%  (43/43)
    ringkasan_fakta      [██████████] 100.0%  (43/43)

  Sumber ekstraksi:
sumber_ekstraksi
regex    43

  Statistik jumlah kata:
    Min  : 62
    Max  : 21,840
    Rata : 7533


## Cell 12 — Preview Hasil

In [36]:
df = pd.read_csv(DATA_PROC / "cases.csv")

print(f"📋 Preview {CONFIG['TOP_K_PREVIEW']} kasus pertama:\n")
for _, row in df.head(CONFIG["TOP_K_PREVIEW"]).iterrows():
    print(f"  [{row['case_id']}]")
    print(f"    No. Perkara   : {row.get('no_perkara', '-')}")
    print(f"    Tanggal       : {row.get('tanggal_putusan', '-')}")
    print(f"    Pengadilan    : {row.get('pengadilan', '-')}")
    print(f"    Terdakwa      : {str(row.get('terdakwa', '-'))[:60]}")
    print(f"    Pasal Dakwaan : {str(row.get('pasal_dakwaan', '-'))[:80]}")
    print(f"    Amar Putusan  : {str(row.get('amar_putusan', '-'))[:80]}")
    print(f"    Vonis         : {row.get('vonis_penjara', '-')}")
    print(f"    Kata kunci PA : {row.get('kata_kunci_pa', '-')}")
    print()

print(f"\n✅ Tahap 2 selesai — lanjut ke 03_retrieval.ipynb")

📋 Preview 5 kasus pertama:

  [case_047]
    No. Perkara   : 37/Pid.Sus-Anak/2016/PN
    Tanggal       : 6 Desember 2016
    Pengadilan    : Pengadilan Negeri Denpasar
    Terdakwa      : ANAK
    Pasal Dakwaan : nan
    Amar Putusan  : terbukti secara sah dan menyakinkan bersalah melakukan Tindak Pidana “ PENCURIAN
    Vonis         : 6 ( enam ) bulan dikurangi sela
    Kata kunci PA : anak, korban

  [case_048]
    No. Perkara   : 14/Pid.Sus.Anak/2015/PN.Dps. hkama ahkamah Agung Repub ahkamah Agung Republik Indonesia mah Agung Republik Indonesia blik Indonesi Anak didampingi pula oleh orang tua anak PK Bapas dan P
    Tanggal       : 27 Agustus 2015
    Pengadilan    : Pengadilan Negeri Denpasar
    Terdakwa      : dalam peradilan tingkat pertama
    Pasal Dakwaan : nan
    Amar Putusan  : terbukti bersalah melakukan Tindak Pidana Narkotika “Secara tanpa hak dan melawa
    Vonis         : nan
    Kata kunci PA : anak, korban, perlindungan, di bawah umur

  [case_049]
    No. Perkara 

---
## ✅ Tahap 2 Selesai!

**Output yang dihasilkan:**
```
/data/processed/
├── cases.csv    ← metadata + fitur (tanpa teks_full)
└── cases.json   ← lengkap dengan teks_full (untuk Tahap 3)
/logs/
└── representation.log
```

**Kolom di `cases.csv`:**
| Kolom | Keterangan |
|-------|-----------|
| case_id | ID unik dokumen |
| no_perkara | Nomor perkara pengadilan |
| tanggal_putusan | Tanggal putusan dijatuhkan |
| pengadilan | Nama pengadilan |
| terdakwa | Nama terdakwa |
| pasal_dakwaan | Pasal yang didakwakan |
| pasal_terbukti | Pasal yang terbukti di putusan |
| amar_putusan | Isi amar putusan singkat |
| vonis_penjara | Lama hukuman penjara |
| vonis_denda | Jumlah denda |
| barang_bukti | Ringkasan barang bukti |
| ringkasan_fakta | Ringkasan fakta hukum |
| jumlah_kata | Jumlah kata dokumen |
| n_kata_kunci_pa | Jumlah kata kunci perlindungan anak |

**Langkah selanjutnya:** Buka `03_retrieval.ipynb` untuk Case Retrieval.